# Doc render (Blender 5.2 on free Colab GPU)
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `kitchen_packed.blend.part* + post.py` to the Files panel (left)
3. Runtime → **Run all**. The finished `doc_render.mp4` downloads at the end.

In [ ]:
SAMPLES = 20     # 40 = same quality as the PC renders
WIDTH   = 720    # 720 -> 720x1280, 1080 for finals
START, END = 1, 945
FPS = 30

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%bash
cd /content
if [ ! -d blender ]; then
  wget -q https://download.blender.org/release/Blender5.2/blender-5.2.2-linux-x64.tar.xz
  tar xf blender-5.2.2-linux-x64.tar.xz && mv blender-5.2.2-linux-x64 blender && rm blender-5.2.2-linux-x64.tar.xz
fi
apt-get -qq install -y libxkbcommon0 libsm6 libxi6 libxrender1 libxxf86vm1 libxfixes3 libgl1 libegl1 > /dev/null 2>&1
./blender/blender --version | head -1

In [ ]:
import glob, os
parts = sorted(glob.glob('/content/*.blend.part*'), key=lambda p: int(p.rsplit('part', 1)[1]))  # numeric: part10 after part9
assert parts, 'Upload the .blend.part files to the Files panel first'
name = os.path.basename(parts[0]).split('.part')[0]
with open('/content/' + name, 'wb') as out:
    for p in parts: out.write(open(p, 'rb').read())
BLEND = '/content/' + name
print('joined', len(parts), 'parts ->', BLEND, os.path.getsize(BLEND), 'bytes')

In [ ]:
%%writefile /content/render.py
# Render a packed .blend on whatever GPU the machine has (Kaggle: 2x Tesla T4), CPU fallback.
# Run: blender -b scene.blend -P kaggle_render.py -- <out_dir> <samples> <width> <start> <end>
import bpy, os, sys, time

argv = sys.argv[sys.argv.index("--") + 1:]
OUT = os.path.abspath(argv[0])
SAMPLES = int(argv[1]); WIDTH = int(argv[2]); START = int(argv[3]); END = int(argv[4])
os.makedirs(OUT, exist_ok=True)

sc = bpy.context.scene
r = sc.render
r.engine = "CYCLES"
sc.cycles.samples = SAMPLES
sc.cycles.use_denoising = True
r.resolution_x = WIDTH
r.resolution_y = int(WIDTH * 16 / 9) // 2 * 2
r.resolution_percentage = 100
r.image_settings.file_format = "PNG"

# ---- use every GPU we can find (OptiX is fastest on NVIDIA, then CUDA)
prefs = bpy.context.preferences.addons["cycles"].preferences
used = None
for backend in ("OPTIX", "CUDA"):
    try:
        prefs.compute_device_type = backend
        prefs.refresh_devices()
        gpus = [d for d in prefs.devices if d.type == backend]
        if gpus:
            for d in prefs.devices:
                d.use = d.type == backend          # GPUs only; CPU would slow the GPUs down
            used = backend
            print("GPU devices:", [d.name for d in gpus], "via", backend, flush=True)
            break
    except Exception as e:
        print("backend", backend, "not available:", e, flush=True)
if used:
    sc.cycles.device = "GPU"
else:
    sc.cycles.device = "CPU"
    print("No GPU found, rendering on CPU", flush=True)
# Kaggle's T4 driver can't create the OptiX denoiser -> always use OpenImageDenoise
sc.cycles.denoiser = "OPENIMAGEDENOISE"
try: sc.cycles.denoising_use_gpu = True
except Exception: pass

def render_to(out):
    r.filepath = out
    bpy.ops.render.render(write_still=True)

t_all = time.time()
for f in range(START, END + 1):
    out = os.path.join(OUT, f"frame_{f:04d}.png")
    if os.path.exists(out):                       # resume after a session cut-off
        continue
    t = time.time()
    sc.frame_set(f)
    try:
        render_to(out)
    except Exception as e:                        # OptiX trouble -> fall back to CUDA, then CPU
        print("render failed:", e, flush=True)
        for backend in ("CUDA", None):
            try:
                if backend:
                    prefs.compute_device_type = backend; prefs.refresh_devices()
                    for d in prefs.devices: d.use = d.type == backend
                    sc.cycles.device = "GPU"
                else:
                    sc.cycles.device = "CPU"
                try: sc.cycles.denoising_use_gpu = False
                except Exception: pass
                render_to(out); print("switched to", backend or "CPU", flush=True); break
            except Exception as e2:
                print("fallback", backend, "failed:", e2, flush=True)
        else:
            raise
    print(f"FRAME {f}/{END}  {time.time() - t:.1f}s", flush=True)
print(f"DONE {END - START + 1} frames in {(time.time() - t_all) / 60:.1f} min", flush=True)


In [ ]:
%%writefile /content/post.py
# "Real camera" pass from the tutorial: softer lens, glow on bright spots, vignette, film grain.
# Usage: python post.py in.png out.png   (same look will be done with ffmpeg for video frames)
import sys, numpy as np
from PIL import Image, ImageFilter

src, dst = sys.argv[1], sys.argv[2]
im = Image.open(src).convert("RGB"); w, h = im.size; s = w / 1080
a = np.asarray(im).astype(np.float32) / 255

# 1. softer lens: blend in a tiny blur
soft = np.asarray(im.filter(ImageFilter.GaussianBlur(0.7 * s))).astype(np.float32) / 255
a = a * 0.65 + soft * 0.35

# 2. glow: blur only what is bright and add it back
lum = a @ np.array([0.2126, 0.7152, 0.0722])
bright = np.clip((lum - 0.72) / 0.28, 0, 1)[..., None] * a
g = Image.fromarray((bright * 255).astype(np.uint8)).filter(ImageFilter.GaussianBlur(18 * s))
a = 1 - (1 - a) * (1 - np.asarray(g).astype(np.float32) / 255 * 0.45)      # screen blend

# 3. vignette
yy, xx = np.mgrid[0:h, 0:w]
d = np.hypot((xx - w / 2) / (w / 2), (yy - h / 2) / (h / 2)) / np.sqrt(2)
a *= (1 - 0.28 * np.clip((d - 0.35) / 0.65, 0, 1) ** 1.6)[..., None]

# 4. film grain (luma noise, slightly stronger in the shadows)
rng = np.random.default_rng(7)
n = rng.normal(0, 1, (h, w)).astype(np.float32)
n = np.asarray(Image.fromarray(((n * 0.25 + 0.5).clip(0, 1) * 255).astype(np.uint8)).filter(ImageFilter.GaussianBlur(0.6 * s))).astype(np.float32) / 255 - 0.5
a += (n * 0.045 * (1.2 - lum.clip(0, 1)))[..., None]

Image.fromarray((a.clip(0, 1) * 255).astype(np.uint8)).save(dst)
print("post ->", dst)


In [ ]:
!cd /content && ./blender/blender -b "{BLEND}" -P render.py -- frames {SAMPLES} {WIDTH} {START} {END} 2>&1 | grep -E 'GPU devices|No GPU|FRAME|DONE|Error|failed|switched'

In [ ]:
import subprocess, glob, os
assert glob.glob('/content/frames/frame_*.png'), 'No frames rendered - see the cell above'
os.makedirs('/content/post', exist_ok=True)
for f in sorted(glob.glob('/content/frames/frame_*.png')):
    subprocess.run(['python', '/content/post.py', f, '/content/post/' + os.path.basename(f)], check=True, capture_output=True)
!ffmpeg -loglevel error -y -framerate {FPS} -start_number {START} -i /content/post/frame_%04d.png -c:v libx264 -pix_fmt yuv420p -crf 17 /content/doc_render.mp4
!ls -lh /content/doc_render.mp4

In [ ]:
from google.colab import files
files.download('/content/doc_render.mp4')